In [2]:
from openai import OpenAI
from itertools import combinations
import json
from collections import defaultdict
from tqdm import tqdm
import random

In [3]:
# init client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
)

In [4]:
# all 9 options
options = [
    "Option A. No businesses are closed; 1 in 100 people will be infected during the next month.",
    "Option B. No businesses are closed; 10 in 100 people will be infected during the next month.",
    "Option C. No businesses are closed; 20 in 100 people will be infected during the next month.",
    "Option D. Crowded, high-risk businesses (such as clubs, theaters, and large sports venues) are closed; 1 in 100 people will be infected during the next month.",
    "Option E. Crowded, high-risk businesses (such as clubs, theaters, and large sports venues) are closed; 10 in 100 people will be infected during the next month.",
    "Option F. Crowded, high-risk businesses (such as clubs, theaters, and large sports venues) are closed; 20 in 100 people will be infected during the next month.",
    "Option G. All non-essential businesses (such as clubs, theaters, restaurants, bars, and gyms) are closed; 1 in 100 people will be infected during the next month.",
    "Option H. All non-essential businesses (such as clubs, theaters, restaurants, bars, and gyms) are closed; 10 in 100 people will be infected during the next month.",
    "Option I. All non-essential businesses (such as clubs, theaters, restaurants, bars, and gyms) are closed; 20 in 100 people will be infected during the next month.",
]

# get all pairwise combinations (72 total)
option_pairs = list(combinations(options, 2))

# roles to test
roles = [
    "public health official",
    "parent",
    "mother",
    "father",
    "student",
    "senior citizen",
    "business owner",
]

In [5]:
# num of full reruns
num_iterations = 5

# structured counter: role -> (pair -> {option: count})
selection_counter = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))

# store full convo logs
conversation_log = []

In [6]:
# iterate over reruns
for run in tqdm(range(num_iterations), desc="Reruns"):
    # vary temperature each run
    temperature = random.uniform(0.3, 1.0)

    # iterate over roles
    for role in roles:
        # iterate over all pairs
        for pair in tqdm(option_pairs, desc=f"Iteration {run+1} ({role})", leave=False):
            prompt_text = (
                f"As a {role}, put yourself back to a time when COVID-19 was spreading in your county and no vaccine was available. "
                "Your county’s public health officials are considering different options for reducing the spread of COVID-19 over the next month. "
                "Of the options shown, which of the following would you choose? Only state the option and no reasoning.\n\n"
                + "\n".join(pair)
            )

            response = client.chat.completions.create(
                model="nvidia/nemotron-nano-9b-v2:free",
                messages=[
                    {"role": "user", "content": [{"type": "text", "text": prompt_text}]}
                ],
                temperature=temperature,
            )

            model_choice = response.choices[0].message.content.strip()

            # update counter
            selection_counter[role][pair][model_choice] += 1

            # append convo
            conversation_log.append({
                "iteration": run + 1,
                "role": role,
                "temperature": temperature,
                "options_presented": pair,
                "model_choice": model_choice
            })

Reruns:   0%|          | 0/5 [04:01<?, ?it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1759795200000'}, 'provider_name': None}}, 'user_id': 'user_33WWyvVS8bNGx9JqCzz4igpmx97'}

In [ ]:
# write full log to JSON
with open("llm_covid_responses_roles.json", "w") as f:
    json.dump(conversation_log, f, indent=2)

# save structured summary
summary = {role: {str(pair): counts for pair, counts in pairs.items()}
           for role, pairs in selection_counter.items()}

with open("llm_covid_selection_summary_roles.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Completed all reruns. Summary written to JSON files.")